# 🎯 [OPS04_M08] 100기 이상 동시 포화 공격 가상 전장 모의 부하 테스트 및 처리 지연 검증
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS04/ops04_m08_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS04_M08 |
| **미션 주제** | 100기 이상 동시 포화 공격 가상 전장 모의 부하 테스트 및 처리 지연 검증 |
| **소요 시간** | 약 3.5시간 |
| **핵심 산출물** | `tests/ops04_stress_test.py, 스트레스 테스트 분석 보고서` |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

## 📂 2. 표준 산출물 디렉토리 체계 및 실습 팁
- **방법 A (`%%writefile` 저장 추천):** 코드 셀 첫 줄에 `%%writefile <산출물 경로>`(예: `%%writefile src/models/ops04_baseline_model.py`)를 추가해 해당 폴더에 모듈을 저장할 수 있습니다.
- **방법 B (셀 단위 논리 대응 인정):** 별도 파일 추출 없이 노트북 셀 단위로 독립 실행·검증하고 미션 10 통합을 완료해도 정상 인정됩니다.

> **⚠️ [교육용 가상 데이터셋 및 HITL 원칙]**
> - 본 노트북의 모든 좌표·비행체 유형·수치는 보안 규정을 준수하여 **교육용 가상 데이터로 생성**됩니다.
> - 실제 무기 제원·사거리·부대 위치·군사 기밀 데이터는 일체 사용하지 않습니다.
> - 산출되는 대응 순위는 지휘관 승인을 위한 **'권고'**이며 자동 승인·자동 실행은 없습니다.

---

In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 100기 이상 동시 포화 공격 가상 전장 모의 부하 테스트 및 처리 지연 검증
**소요시간**: 약 3.5시간 | **이번 산출물**: `tests/ops04_stress_test.py, 스트레스 테스트 분석 보고서`

### 📋 미션 개요
**100기에서 600기(500% 증가)까지 동시 항적이 급증하고 손상·중복·순서 역전 메시지가 유입되는 극한 포화 상황에서 시스템 무중단과 데이터 무유실을 검증합니다.**

- **배경 및 필요성:** 평시 표본에서 정상 동작하던 파이프라인도 대규모 동시 포화 공격 시 메모리 병목이나 처리 시간 예산 초과로 멈출 수 있어, 단계적 부하 시험과 기능 축소(Graceful Degradation) 복원력 검증이 필수적입니다.
- **핵심 과업:** 100기→600기 단계별 부하와 손상·중복·순서 뒤섞임을 주입해 모든 메시지를 결과·격리·중복으로 완전 계수하고, 처리 시간 예산 초과 시 도달시간 순 정렬만 유지하는 기능 축소 모드 전환 및 긴급 표적 누락 여부를 시험합니다.
- **학습 목표 (습득 역량):** 대량 트래픽 스트레스 테스트 시나리오 설계 역량, 메시지 완전 계수(Accounting) 기반 무유실 검증 능력, 시간 예산 초과에 대비한 Fail-safe 기능 축소 아키텍처 구현 역량을 습득합니다.
- **검증 및 산출물:** 동시 트래픽 500% 증가 시 시스템 다운 0건, 데이터 유실률 0%, 긴급 표적 누락 0건을 입증한 부하 테스트 코드(`tests/ops04_stress_test.py`)와 스트레스 테스트 분석 보고서를 산출합니다.

### 🎓 학습 목표 (습득 역량)
**[습득 역량]** 평시 대비 500% 급증하는 동시 포화 공격과 복합 데이터 결함 환경에서 시스템 병목, 데이터 무유실, 시간 예산 초과 시 Fail-safe 기능 축소 동작을 검증하는 부하 테스트 엔지니어링 역량을 체득합니다.

**[핵심 과업]** 100기에서 600기까지 부하를 단계적으로 늘리며 손상 메시지·중복·순서 뒤섞임을 주입해 결과·격리·중복 완전 계수로 유실률 0%를 확인하고, 시간 예산 초과 시 도달시간 순 정렬만 수행하는 기능 축소 모드 전환을 시험합니다.

- **극한 포화 부하 및 복합 결함 시나리오 설계 역량:** 표적 수 5배 급증(100→600기)과 손상·중복·순서 역전 패킷이 동시에 쏟아지는 스트레스 환경을 모의해 처리 지연 임계점을 식별하는 능력을 익힙니다.
- **완전 계수(Accounting) 기반 데이터 무유실 검증 능력:** 유입된 모든 메시지를 정상 결과·격리·중복 제거 중 하나로 100% 추적 계수해 포화 상황에서도 시스템 다운 0건·데이터 유실률 0%를 입증하는 검증 기술을 습득합니다.
- **시간 예산 기반 Fail-safe 기능 축소 아키텍처 역량:** 연산 예산 초과 시 고비용 보조 계산을 중단하고 핵심 도달시간(TTI) 순 정렬로 자동 전환해 긴급 표적 누락 0건을 보장하는 복원력(Resilience) 설계 원리를 체득합니다.

### 📖 미션 배경 및 시나리오
#### 미션 배경 및 당위성
수백 발의 드론/미사일 동시 유입 모의 스트레스 테스트를 수행하여 시스템 병목 구간을 프로파일링하고 실시간 처리 성능을 검증한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **100기 이상이 동시에 들어오는 포화 상황에서 시스템이 멈추거나 데이터를 잃지 않는지 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **평시 100기에서 600기(500% 증가)까지 단계적으로 늘리며 손상 메시지·중복·순서 뒤섞임 주입하여 처리 지연 및 복원력을 시험하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `tests/ops04_stress_test.py 및 스트레스 테스트 분석 보고서` 형태의 결과물을 완성한다.

### 🤖 단계별 수행 및 AI 협업 가이드
#### Step 1. 미션 8 코드를 실행해 부하 단계별 처리시간·격리·중복·유실을 확인한다.
#### 이 단계의 역할
미션 8의 `run_stage()` 부하 테스트를 실행해 100기에서 600기(500% 증가)까지 단계별 처리시간·격리·중복·유실률을 확인합니다.

#### 해야 할 일

- 평시 100기에서 최대 600기까지 500% 부하가 증가할 때 `inject()`가 주입하는 손상·중복·순서 역전 메시지 구성을 파악한다.
- AI를 Co-pilot으로 활용해 각 부하 단계에서 `입력 총수 == 정상 처리 + 격리 + 중복` 항등식이 성립하는지 검증하는 계수 로직을 점검한다.
- 시스템 다운 0건, 데이터 유실률 0.00% 달성 여부를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "100기에서 600기(500% 증가)까지 단계별 부하와 손상·중복·순서 역전을 주입하는 run_stage()에서, 모든 부하 단계마다 처리시간(ms), 격리 건수, 중복 건수, 유실률(입력 - (결과+격리+중복))이 0.00%인지 검증하는 코드를 검토해줘."

  - **AI 결과 검증 포인트:** AI가 유실률을 계산할 때 중복 메시지나 격리 메시지를 누락 분모에서 빼버리는 왜곡이 없는지, 메시지 ID 단위 완전 계수 항등식이 성립하는지 확인합니다.

#### 핵심 확인 지표

> 100->600기(500% 증가) 시스템 다운 0건, 데이터 유실률 0.00%

#### 학습자가 제출해야 하는 내용

- 미션 8 부하 단계별(100~600기) 처리시간·격리·중복·유실 로그
- AI 프롬프트 질의 및 완전 계수 항등식 검증 기록

#### Step 2. Fail-safe 시험에서 기능 축소 전환과 긴급 표적 누락 여부를 본다.
#### 이 단계의 역할
시간 예산(`BUDGET_MS = 0.1ms`) 강제 축소 시 기능 축소(Fail-safe) 모드로 안전하게 전환되고 긴급 표적 누락이 0건인지 확인합니다.

#### 해야 할 일

- `process()` 함수에서 처리 시간이 `BUDGET_MS`를 초과할 때 고비용 보조 평가 대신 도달시간(TTI) 순 정렬로 폴백하는 분기를 확인한다.
- AI에게 예산 `0.1ms` 강제 상황에서 기능 축소 전환 여부와 실제 긴급 표적 197건의 누락 건수를 대조하는 테스트 코드를 요청한다.
- 기능 축소 전환 로그와 긴급 표적 197건 중 누락 0건 결과를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "부하 처리 예산(BUDGET_MS)을 0.1ms로 강제 제한했을 때 Fail-safe 기능 축소 모드로 전환되는지, 그리고 기능 축소 중에도 실제 긴급 표적 197건이 우선순위 대기열에서 누락되지 않고(누락 0건) 유지되는지 검증하는 코드를 작성해줘."

  - **AI 결과 검증 포인트:** 기능 축소 전환 시 예외가 발생해 프로세스가 중단되거나, 긴급 표적이 큐 끝으로 밀려나 누락되지 않았는지 실제 긴급 ID 집합 대조로 확인합니다.

#### 핵심 확인 지표

> 예산 0.1ms 강제 시 Fail-safe 기능 축소 전환, 긴급 표적 197건 중 누락 0건

#### 학습자가 제출해야 하는 내용

- Fail-safe 기능 축소 전환 로그 및 긴급 표적 197건 누락 0건 검증 결과
- AI 작성 Fail-safe 테스트 검증 기록

#### Step 3. 그래프에서 처리시간이 예산선 아래인지 확인한다.
#### 이 단계의 역할
부하 단계별 처리시간 그래프에서 모든 단계가 처리시간 예산선(`BUDGET_MS`) 아래에 머무는지 프로파일링합니다.

#### 해야 할 일

- 100기~600기 부하 단계별 처리시간 추이 그래프를 확인하고 선형/초선형 증가 추세를 점검한다.
- AI에게 600기 구간에서 처리시간이 가장 많이 소요되는 함수 병목을 프로파일링하고 `스트레스 테스트 분석 보고서` 요약 문단을 작성해 달라고 요청한다.
- 처리시간 예산선 여유율과 메모리/지연 안정성 결론을 정리한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "100기~600기 부하 단계별 처리시간(ms) 그래프 데이터와 예산선 대비 여유율을 분석해, 병목 가능 지점과 Fail-safe 복원력 결론을 담은 스트레스 테스트 분석 보고서 요약을 작성해줘."

  - **AI 결과 검증 포인트:** AI가 작성한 보고서의 처리시간 수치(ms)와 증가율(500%)이 실제 실행 출력과 정확히 일치하는지 대조합니다.

#### 핵심 확인 지표

> 전 부하 단계(100~600기) 처리시간 예산선 이하 유지 및 분석 보고서 완성

#### 학습자가 제출해야 하는 내용

- 부하 단계별 처리시간 그래프 및 스트레스 테스트 분석 보고서
- AI 작성 보고서 수치 대조 검증 기록

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
`inject()`의 손상 메시지 비율을 `5%`에서 `30%`로 대폭 높였을 때 격리 건수와 데이터 유실률 변화를 검증합니다.

#### 해야 할 일

- `inject()`의 손상 비율 파라미터를 `0.05`에서 `0.30`으로 올려 부하 테스트를 재실행한다.
- AI에게 손상 30% 극한 환경에서 격리 건수 급증에도 유실률 0.00%와 시스템 무중단이 유지되는 이유를 정리하는 비교 분석을 요청한다.
- 손상 비율 5% vs 30% 비교표와 Fail-safe 안정성 결론을 작성한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "inject()의 손상 메시지 비율을 5%에서 30%로 높여 실행했을 때 정상 처리 건수, 격리 건수, 데이터 유실률 변화를 비교표로 출력하고 극한 결함 환경에서의 시스템 무결성을 평가해줘."

  - **AI 결과 검증 포인트:** 손상 비율 30%에서도 파싱 예외로 파이프라인 전체가 멈추지 않고 손상 건만 정확히 격리 계수되어 유실률 0.00%가 유지되는지 확인합니다.

#### 핵심 확인 지표

> 손상 비율 5% vs 30% 비교, 시스템 다운 0건 및 데이터 유실률 0.00% 유지

#### 학습자가 제출해야 하는 내용

- 손상 비율 변경 전후 비교 분석 메모 및 테스트 로그
- AI 질의 프롬프트 및 예외 격리 무결성 검증 기록

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가영역:** 극단 상황에서의 안전 실패(Fail-safe) 및 기능 축소 작동 적절성

#### 미션 PASS 조건
- 정량 기준: 동시 트래픽 500% 증가 시 시스템 다운 0건, 데이터 유실률 0%
- 진행 기준: 대량 시뮬레이션 부하 주입 테스트 통과 및 병목 구간 프로파일링 완료
- 검증 방법: 부하 테스트 도구를 통한 처리량(Throughput) 및 메모리 누수 검증
- AI 검증 기준: 포화 부하 완전 계수 항등식과 0.1ms 예산 Fail-safe 긴급 표적 무누락 동작을 AI 협업 코드 리뷰 및 스트레스 시험으로 입증 완료
#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 AI 프롬프트 설계의 구체성, AI 산출물의 오류·보안 위반을 인간 검토(HITL)로 식별·수정한 기록, 그리고 출력 수치의 군 실무 부합성을 종합 채점한다.

In [ ]:
# [TODO : 학생 실습 8] 핵심 함수의 TODO 영역을 AI Co-pilot과 함께 구현하고 검증하세요.
"""평시 100기 기준에서 최대 600기(500% 증가)까지 가상 비행체가 동시에 유입되는 포화 공격 상황에 손상 메시지·중복·순서 뒤섞임을 주입해 부하 시험을 수행하고, 단계별 처리 지연·시스템 다운·데이터 유실과 시간 예산 초과 시 기능 축소(Fail-safe) 동작을 검증한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------- [교육용가상자료] 가상 방어지역·가상 비행체 생성기 ----------------
# 좌표는 가상 지휘소를 원점으로 한 km 단위 평면이다. 비행체 유형은 추상 범주이며
# 속도·고도·반사강도는 학습용 임의 범위로, 실제 무기 제원·사거리·탄도와 무관하다.
ASSETS = pd.DataFrame([["A1", "CP(virtual)", 0.0, 0.0, 1.0],
                       ["A2", "Radar(virtual)", 4.0, -3.0, 0.8],
                       ["A3", "Depot(virtual)", -5.0, 4.0, 0.7],
                       ["A4", "Power(virtual)", 7.0, 6.0, 0.5]], columns=["asset", "name", "x", "y", "value"])
TYPES = {  # 유형: (위협 여부, 속도 m/s 범위, 고도 m 범위, 출현 비율)
    "uav_type": (True, (30, 60), (100, 800), 0.42),      # 저속 무인기형(가상)
    "fast_type": (True, (150, 250), (50, 200), 0.16),    # 고속 순항형(가상)
    "decoy": (False, (30, 60), (100, 800), 0.16),        # 기만체(가상)
    "civil": (False, (120, 220), (3000, 9000), 0.10),    # 민항기(가상)
    "bird": (False, (8, 18), (50, 400), 0.16),           # 조류 군집
}
TTI_URGENT = 600      # 긴급 판정 기준: 가상 자산 도달 예상시간 600초 이내
CPA_KM = 1.5          # 자산 상공 통과 판정 반경(가상)


def make_raid(seed, n=80, spoof=0):
    """가상 포화 상황 1회분(비행체 n개 + 스푸핑 가짜 항적 spoof개)을 만든다."""
    rng = np.random.default_rng(seed)
    names = list(TYPES)
    rows = []
    for i in range(n + spoof):
        typ = "spoof" if i >= n else str(rng.choice(names, p=[TYPES[k][3] for k in names]))
        threat, sp, al, _ = TYPES["decoy" if typ == "spoof" else typ]
        brg = np.radians(rng.uniform(0, 360)); dist = rng.uniform(6, 60)
        x, y = dist * np.sin(brg), dist * np.cos(brg)
        spd = rng.uniform(*sp); alt = rng.uniform(*al)
        aim = int(rng.integers(0, len(ASSETS)))
        if typ in ("uav_type", "fast_type", "decoy", "spoof"):      # 가상 자산 방향(작은 방위 오차)
            ax_, ay_ = ASSETS.loc[aim, ["x", "y"]]
            hd = np.arctan2(ax_ - x, ay_ - y) + np.radians(rng.normal(0, 0.6))
        else:
            hd = rng.uniform(0, 2 * np.pi); aim = -1
        vx, vy = spd * np.sin(hd) / 1000, spd * np.cos(hd) / 1000          # km/s
        sig = {"uav_type": ((0.2, 0.5), (0.02, 0.12), [0, .1, .5, .4]), "fast_type": ((0.1, 0.4), (0.02, 0.12), [0, .2, .4, .4]),
               "decoy": ((0.3, 0.8), (0.08, 0.35), [0, .55, .4, .05]), "spoof": ((0.3, 0.8), (0.05, 0.3), [0, .7, .3, 0]),
               "civil": ((0.7, 1.0), (0.01, 0.05), [0, 0, .1, .9]), "bird": ((0.01, 0.12), (0.1, 0.4), [0, .6, .4, 0])}[typ]
        jump = spd * rng.uniform(0.95, 1.2) if typ != "spoof" else spd * rng.uniform(4, 15)
        true_tti = np.inf
        if threat:
            d_aim = np.hypot(ASSETS.loc[aim, "x"] - x, ASSETS.loc[aim, "y"] - y)
            true_tti = d_aim * 1000 / spd
        rows.append({"id": f"T{seed % 100:02d}{i:03d}", "type": typ, "x": x, "y": y, "vx": vx, "vy": vy, "alt": alt, "spd": spd,
                     "rcs": rng.uniform(*sig[0]), "rcs_var": rng.uniform(*sig[1]), "n_sensors": int(rng.choice(4, p=sig[2])),
                     "max_jump": jump, "civ_iff": int(typ == "civil" and rng.random() < 0.95),
                     "aim": aim, "threat": threat, "true_tti": true_tti,
                     "urgent": bool(threat and true_tti <= TTI_URGENT)})
    df = pd.DataFrame(rows)
    df["true_prio"] = [ASSETS.loc[a, "value"] * np.exp(-t / 300) if th else 0.0
                       for a, t, th in zip(df["aim"], df["true_tti"], df["threat"])]
    return df


def geometry(df):
    """각 항적과 가상 자산 사이 최근접점(CPA) 거리와 도달 예상시간(TTI, s)을 계산한다."""
    px, py = df["x"].to_numpy()[:, None], df["y"].to_numpy()[:, None]
    vx, vy = df["vx"].to_numpy()[:, None], df["vy"].to_numpy()[:, None]
    rx, ry = ASSETS["x"].to_numpy()[None] - px, ASSETS["y"].to_numpy()[None] - py
    v2 = vx ** 2 + vy ** 2 + 1e-12
    t = (rx * vx + ry * vy) / v2                                  # CPA 도달 시각(s)
    cpa = np.hypot(rx - vx * t, ry - vy * t)
    tti = np.where((t > 0) & (cpa < CPA_KM), t, np.inf)
    return cpa, tti

# ---------------- 위협 점수 모델(추상 모델: 거리·도달시간·자산 가치·위협 유사도) ----------------
FEATS = ["log_spd", "alt_km", "rcs", "rcs_var", "n_sensors", "civ_iff", "jump_ratio"]


def features(df):
    return pd.DataFrame({"log_spd": np.log(df["spd"]), "alt_km": df["alt"] / 1000, "rcs": df["rcs"], "rcs_var": df["rcs_var"],
                         "n_sensors": df["n_sensors"], "civ_iff": df["civ_iff"], "jump_ratio": df["max_jump"] / df["spd"]})


def train_classifier(seeds=(101, 102, 103, 104)):
    """학습용 가상 상황(시험 상황과 다른 시드)으로 '위협 유사도' 분류기를 학습한다."""
    tr = pd.concat([make_raid(s, 120, spoof=10) for s in seeds], ignore_index=True)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=1.0))
    clf.fit(features(tr)[FEATS], tr["threat"].astype(int))
    return clf


def baseline_rank(df):
    """Baseline: 거리·도달시간만으로 1차 위협 점수를 매긴다(자산 가치·유형 판별 없음)."""
    _, tti = geometry(df)
    t_min = tti.min(axis=1)
    out = df[["id"]].copy()
    out["tti_s"] = t_min
    out["score"] = np.exp(-t_min / 300)
    out["urgent_pred"] = t_min <= TTI_URGENT
    return out.sort_values("score", ascending=False)


def advanced_rank(df, clf, w=None):
    """고도화: 자산 가치 x 도달시간 x 위협 유사도, 운동학 불일치(스푸핑 의심) 항적은 별도 표시한다."""
    w = w or {"tau": 300, "p_min": 0.5, "value_pow": 1.0, "jump_max": 2.0}
    _, tti = geometry(df)
    val = ASSETS["value"].to_numpy()[None] ** w["value_pow"]
    asset_score = val * np.exp(-tti / w["tau"])
    best = asset_score.argmax(axis=1)
    p = clf.predict_proba(features(df)[FEATS])[:, 1]
    spoof = (df["max_jump"] / df["spd"]).to_numpy() > w["jump_max"]
    out = df[["id"]].copy()
    out["asset"] = ASSETS["asset"].to_numpy()[best]
    out["tti_s"] = tti[np.arange(len(df)), best]
    out["p_threat"] = p
    out["spoof_flag"] = spoof
    out["score"] = np.where(spoof, 0, p * asset_score.max(axis=1))
    out["urgent_pred"] = (p >= w["p_min"]) & (out["tti_s"] <= TTI_URGENT) & ~spoof
    return out.sort_values("score", ascending=False)

BUDGET_MS = 200          # 1회 처리 시간 예산
clf = train_classifier()


def inject(df, rng):
    """손상(NaN·음수 속도) 5%, 중복 10%, 순서 뒤섞기"""
    d = df.copy()
    bad = rng.random(len(d)) < 0.05
    d.loc[bad, "spd"] = rng.choice([np.nan, -1.0], bad.sum())
    dup = d.sample(frac=0.10, random_state=int(rng.integers(1e6)))
    return pd.concat([d, dup]).sample(frac=1, random_state=1).reset_index(drop=True)


def process(msgs, degrade=False):
    """[TODO : 학생 실습 8-1] 검증->격리->중복제거->순위 산정을 수행하며 degrade=True면 baseline_rank만 호출하세요."""
    # 힌트: 모든 유입 메시지가 ranked + quarantine + dup로 완전 계수되어야 유실률 0% 검증이 통과됩니다.
    pass  # <<< 여기에 process() 로직을 작성한 뒤 pass를 제거하세요 >>>


def run_stage(msgs, budget_ms):
    """[TODO : 학생 실습 8-2] process(msgs) 처리 시간이 budget_ms를 넘으면 degrade=True(Fail-safe)로 재처리하세요."""
    pass  # <<< 여기에 run_stage() 로직을 작성한 뒤 pass를 제거하세요 >>>


rows = []
rng = np.random.default_rng(8)
for n in (100, 200, 300, 450, 600):
    raid = make_raid(30 + n, n, spoof=n // 20)
    msgs = inject(raid, rng)
    crash = 0
    try:
        (ranked, nq, ndup), ms, degraded = run_stage(msgs, BUDGET_MS)
    except Exception:               # 다운 계수(발생 시에도 다음 단계 계속)
        crash, ranked, nq, ndup, ms, degraded = 1, pd.DataFrame(columns=["id"]), 0, 0, np.nan, False
    accounted = len(ranked) + nq + ndup
    expect_ids = set(msgs.loc[msgs["spd"].notna() & (msgs["spd"] > 0), "id"])
    lost = len(expect_ids - set(ranked["id"]))
    rows.append({"비행체": n, "증가율": f"{(n / 100 - 1):.0%}", "메시지": len(msgs), "처리(ms)": round(ms, 1), "격리": nq, "중복": ndup,
                 "계수누락": len(msgs) - accounted, "유실": lost, "다운": crash, "기능축소": degraded})
res = pd.DataFrame(rows)
print("[부하 단계별 결과]")
print(res.to_string(index=False))

# Fail-safe 확인: 예산을 0.1 ms로 강제해 기능 축소 경로가 실제로 동작하는지 시험
raid = make_raid(99, 600, spoof=30); msgs = inject(raid, rng)
(r_deg, nq, _), ms, degraded = run_stage(msgs, 0.1)
valid = set(msgs.loc[msgs["spd"].notna() & (msgs["spd"] > 0), "id"])
urgent_ids = set(raid.loc[raid["urgent"], "id"]) & valid
miss = len(urgent_ids - set(r_deg.loc[r_deg["urgent_pred"], "id"]))
print(f"\n[Fail-safe 시험] 예산 0.1 ms 강제 -> 기능 축소 {'전환' if degraded else '미전환'}, 긴급 표적 {len(urgent_ids)}건 중 누락 {miss}건")
print(f"  기능 축소 모드는 도달시간 기준 경보를 유지하고(누락 방지 우선), 늘어난 오경보와 격리 메시지 {nq}건은 운용자 확인으로 처리한다.")
print("병목 구간: 분류기 추론·자산별 CPA 행렬 계산(항적 수 x 자산 수)에 비례, 600기에서도 예산 내")

down, loss = int(res["다운"].sum()), int(res["유실"].sum() + res["계수누락"].sum())
print(f"\n트래픽 500% 증가 시 시스템 다운 {down}건 (목표 0건) -> {'달성' if down == 0 else '미달'}")
print(f"데이터 유실 {loss}건, 유실률 {loss / res['메시지'].sum():.2%} (목표 0%) -> {'달성' if loss == 0 else '미달'}")

plt.figure(figsize=(7, 4))
plt.plot(res["비행체"], res["처리(ms)"], "o-", label="processing time")
plt.axhline(200, ls="--", c="r", label="budget 200 ms")
plt.xlabel("Simultaneous tracks"); plt.ylabel("ms"); plt.title("Stress test: latency vs load (virtual)"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[부하 단계별 결과]
 비행체  증가율  메시지  처리(ms)  격리  중복  계수누락  유실  다운  기능축소
 100   0%  115     9.8   2  10     0   0   0 False
 200 100%  231     9.8   9  20     0   0   0 False
 300 200%  347    11.8   9  31     0   0   0 False
 450 350%  519    14.8  27  46     0   0   0 False
 600 500%  693    10.9  28  60     0   0   0 False

[Fail-safe 시험] 예산 0.1 ms 강제 -> 기능 축소 전환, 긴급 표적 197건 중 누락 0건
  기능 축소 모드는 도달시간 기준 경보를 유지하고(누락 방지 우선), 늘어난 오경보와 격리 메시지 23건은 운용자 확인으로 처리한다.
병목 구간: 분류기 추론·자산별 CPA 행렬 계산(항적 수 x 자산 수)에 비례, 600기에서도 예산 내

트래픽 500% 증가 시 시스템 다운 0건 (목표 0건) -> 달성
데이터 유실 0건, 유실률 0.00% (목표 0%) -> 달성
```
</details>
